In [ ]:
# Import libraries

import csv
import hashlib
import importlib.metadata
import json
import os
import random
import shutil
import sys
import time
from datetime import datetime
from pathlib import Path

CACHE_DIR = Path("pretrained") / "smp_cache"
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import segmentation_models_pytorch as smp
    import timm
    import torchvision
except ImportError as error:
    raise RuntimeError(
        "Required packages are missing. Install segmentation-models-pytorch 0.5.0, "
        "timm, and torchvision in the selected Python environment."
    ) from error

cv2.setNumThreads(0)
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select an environment with CUDA-enabled PyTorch.")
if smp.__version__ != "0.5.0":
    raise RuntimeError(f"This notebook requires segmentation-models-pytorch 0.5.0; found: {smp.__version__}")

DEVICE = torch.device("cuda:0")
VERSIONS = {
    "python": sys.version.split()[0], "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__), "smp": str(smp.__version__),
    "timm": str(timm.__version__),
    "huggingface_hub": importlib.metadata.version("huggingface-hub"),
    "numpy": str(np.__version__), "cuda_runtime": torch.version.cuda,
}
print("Python:", sys.executable)
print("Package versions:", VERSIONS)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("SMP location:", smp.__file__)
import copy


In [ ]:
# Configure V2 paths and training settings

MODEL_NAME = "MiT-B2_U-Net"
ENCODER_NAME = "mit_b2"
ENCODER_WEIGHTS = "imagenet"
EXPERIMENT_NAME = "V2"
DATA_ROOT = Path("data") / "V2"
OUTPUT_DIR = Path("results") / "MiT_B2_UNet" / "V2"

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
EPOCHS = 50
LEARNING_RATE = 1e-4
THRESHOLD = 0.5
TRAINING_SEED = 145
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
SAVE_EVERY_EPOCH = True
EXPECTED_COUNTS = {"Train": 967, "Validation": 276, "Test": 138}

assert DATA_ROOT.is_dir(), f"Dataset directory not found: {DATA_ROOT}"
assert EPOCHS == 50 and SAVE_EVERY_EPOCH
print("Dataset:", DATA_ROOT)
print("Output:", OUTPUT_DIR)


In [ ]:
# Define the dataset

IMAGE_EXTENSIONS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class BuildingPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.images = {
            path.name: path for path in self.image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        self.masks = {
            path.name: path for path in self.mask_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        if self.images.keys() != self.masks.keys():
            only_images = sorted(self.images.keys() - self.masks.keys())[:5]
            only_masks = sorted(self.masks.keys() - self.images.keys())[:5]
            raise ValueError(
                f"Image and mask filenames do not match. "
                f"Images without masks: {only_images}; masks without images: {only_masks}"
            )
        self.names = sorted(self.images)
        if not self.names:
            raise ValueError(f"No images found: {self.image_dir}")

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image_bgr = cv2.imread(str(self.images[name]), cv2.IMREAD_COLOR)
        raw_mask = cv2.imread(str(self.masks[name]), cv2.IMREAD_GRAYSCALE)
        if image_bgr is None or raw_mask is None:
            raise RuntimeError(f"Cannot read file: {name}")
        if image_bgr.shape[:2] != raw_mask.shape:
            raise ValueError(f"Image and mask dimensions do not match: {name}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)))


        mask = (raw_mask == 0).astype(np.float32)[None, ...]
        mask = torch.from_numpy(mask)
        return image, mask


In [ ]:
# Check image and mask pairs

DATASETS = {EXPERIMENT_NAME: {}}
for split, expected in EXPECTED_COUNTS.items():
    dataset = BuildingPatchDataset(
        DATA_ROOT / split / "Image", DATA_ROOT / split / "Mask"
    )
    if len(dataset) != expected:
        raise ValueError(
            f"{split}: expected {expected} patches, found {len(dataset)}."
        )
    DATASETS[EXPERIMENT_NAME][split] = dataset
    print(f"{EXPERIMENT_NAME}/{split}: {len(dataset)} patches")

parents = {
    split: {Path(name).stem.rsplit("_", 1)[0] for name in dataset.names}
    for split, dataset in DATASETS[EXPERIMENT_NAME].items()
}
for first, second in (("Train", "Validation"), ("Train", "Test"), ("Validation", "Test")):
    overlap = parents[first] & parents[second]
    if overlap:
        raise ValueError(f"Parent-tile overlap {first}/{second}: {sorted(overlap)[:5]}")
print("Parent-tile counts:", {name: len(ids) for name, ids in parents.items()})
print("Parent-tile intersections: empty")

sample_dataset = DATASETS[EXPERIMENT_NAME]["Train"]
sample_name = sample_dataset.names[0]
sample_image, sample_target = sample_dataset[0]
if tuple(sample_image.shape) != (3, 512, 512):
    raise ValueError(f"Expected input (3,512,512); found {tuple(sample_image.shape)}")
raw_mask = cv2.imread(str(sample_dataset.masks[sample_name]), cv2.IMREAD_GRAYSCALE)
rgb = sample_image.permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.clip(rgb, 0.0, 1.0))
axes[1].imshow(raw_mask, cmap="gray", vmin=0, vmax=255)
axes[2].imshow(sample_target[0].numpy(), cmap="gray", vmin=0, vmax=1)
for ax, title in zip(axes, ("Image", "Raw mask", "Metric target: raw == 0")):
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(sample_name)
fig.tight_layout()
plt.show()
plt.close(fig)
print("Raw mask values:", np.unique(raw_mask))
print("Reference positive target: raw_mask == 0; the mapping is unchanged.")


In [ ]:
# Define and check the model

def build_model(pretrained=False):
    return smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS if pretrained else None,
        in_channels=3,
        classes=1,
        activation=None,
    )



with torch.random.fork_rng(devices=[DEVICE]):
    probe_model = build_model(pretrained=True).to(DEVICE)
    probe_model.eval()
    with torch.no_grad():
        probe_input = torch.zeros(BATCH_SIZE, 3, 512, 512, device=DEVICE)
        probe_output = probe_model(probe_input)
    expected_shape = (BATCH_SIZE, 1, 512, 512)
    if tuple(probe_output.shape) != expected_shape:
        raise RuntimeError(f"Expected output {expected_shape}; found {tuple(probe_output.shape)}")
    if not torch.isfinite(probe_output).all():
        raise RuntimeError("The model check produced non-finite outputs.")
    MODEL_INFO = {
        "architecture": MODEL_NAME, "encoder_name": ENCODER_NAME,
        "encoder_weights": ENCODER_WEIGHTS,
        "parameters": sum(p.numel() for p in probe_model.parameters()),
        "trainable_parameters": sum(p.numel() for p in probe_model.parameters() if p.requires_grad),
        "input_shape": [BATCH_SIZE, 3, 512, 512],
        "output_shape": list(probe_output.shape),
        "batchnorm_layers": sum(isinstance(m, nn.BatchNorm2d) for m in probe_model.modules()),
    }
    del probe_model, probe_input, probe_output
torch.cuda.empty_cache()
print("ImageNet weights loaded; GPU output check passed.")
print("Model information:", MODEL_INFO)


In [ ]:
# Define Dice loss and patch metrics

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probabilities = torch.sigmoid(logits).flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)
    intersection = (probabilities * targets).sum(dim=1)
    denominator = probabilities.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def patch_metrics(logits, targets):
    predictions = torch.sigmoid(logits) >= THRESHOLD
    targets = targets >= 0.5
    predictions = predictions.flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)

    tp = (predictions & targets).sum(dim=1).float()
    fp = (predictions & ~targets).sum(dim=1).float()
    fn = (~predictions & targets).sum(dim=1).float()
    tn = (~predictions & ~targets).sum(dim=1).float()
    eps = 1e-7

    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "f1": (2 * tp) / (2 * tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }


In [ ]:
# Define the epoch routine

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

METRIC_NAMES = ("accuracy", "iou", "f1", "precision", "recall")


def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    totals = {"loss": 0.0, **{name: 0.0 for name in METRIC_NAMES}}
    sample_count = 0

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        batch_size = images.shape[0]

        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = dice_loss_with_logits(logits, masks)
            if is_training:
                loss.backward()
                optimizer.step()

        batch_metrics = patch_metrics(logits.detach(), masks)
        totals["loss"] += loss.detach().item() * batch_size
        for name, values in batch_metrics.items():
            totals[name] += values.sum().item()
        sample_count += batch_size

    if sample_count == 0:
        raise RuntimeError("The DataLoader is empty.")
    return {name: value / sample_count for name, value in totals.items()}


In [ ]:
# Create data loaders

def make_loaders(split_datasets):
    generator = torch.Generator().manual_seed(TRAINING_SEED)
    return {
        "Train": DataLoader(
            split_datasets["Train"], batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False, generator=generator,
        ),
        "Validation": DataLoader(
            split_datasets["Validation"], batch_size=EVAL_BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False,
        ),
        "Test": DataLoader(
            split_datasets["Test"], batch_size=EVAL_BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False,
        ),
    }


In [ ]:
# Define checkpoint and BN calibration helpers

WEIGHTS_DIR = OUTPUT_DIR / "weights"
EPOCH_WEIGHTS_DIR = WEIGHTS_DIR / "epochs"
PREDICTION_DIR = OUTPUT_DIR / "prediction"
MASK_DIR = PREDICTION_DIR / "masks"
CONFUSION_DIR = OUTPUT_DIR / "confusion"
BEST_PATH = WEIGHTS_DIR / "best_model.pth"
FINAL_PATH = WEIGHTS_DIR / "final_model.pth"
HISTORY_PATH = OUTPUT_DIR / "history.json"
TRAIN_CSV_PATH = OUTPUT_DIR / "train_logs.csv"


def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)


def write_rows(path, rows):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def write_training_history(history):
    names = (("loss", "loss"), ("accuracy", "accuracy"), ("iou", "iou_score"),
             ("f1", "fscore"), ("precision", "precision"), ("recall", "recall"))
    rows = []
    for record in history:
        row = {"epoch": record["epoch"]}
        for key, column in names:
            row[column] = record["train"][key]
        for key, column in names:
            row["val_" + column] = record["validation"][key]
        for key, column in names:
            row["normal_val_" + column] = record["normal_validation"][key]
        rows.append(row)
    write_rows(TRAIN_CSV_PATH, rows)
    save_json(HISTORY_PATH, history)


def save_model_checkpoint(path, model, epoch, validation, best_val_iou):
    payload = {
        "epoch": epoch, "experiment": EXPERIMENT_NAME,
        "architecture": MODEL_NAME, "encoder_name": ENCODER_NAME,
        "bn_calibrated": True, "validation": validation,
        "best_val_iou": best_val_iou,
        "model_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
    }
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def checkpoint_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


print("Weights:", WEIGHTS_DIR)
print("Prediction:", PREDICTION_DIR)
print("Confusion:", CONFUSION_DIR)


def recalibrate_bn_copy(source_model, calibration_loader):
    calibrated_model = copy.deepcopy(source_model).to(DEVICE)
    calibrated_model.eval()
    bn_layers = [m for m in calibrated_model.modules() if isinstance(m, nn.BatchNorm2d)]
    if not bn_layers:
        raise RuntimeError("Modelde No BatchNorm2d layers found.")
    for module in bn_layers:
        module.reset_running_stats()
        module.momentum = None
        module.train()

    with torch.no_grad():
        for images, _ in calibration_loader:
            calibrated_model(images.to(DEVICE, non_blocking=True))
    calibrated_model.eval()
    return calibrated_model


In [ ]:
# Define the training loop

def train_calibrated_experiment():
    config = {
        "experiment": EXPERIMENT_NAME, "data_root": str(DATA_ROOT),
        "architecture": MODEL_NAME, "encoder_name": ENCODER_NAME,
        "encoder_weights": ENCODER_WEIGHTS,
        "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "evaluation_batch_size": EVAL_BATCH_SIZE,
        "optimizer": "RMSprop", "learning_rate": LEARNING_RATE,
        "loss": "Dice loss", "threshold": THRESHOLD,
        "seed": TRAINING_SEED, "augmentation": "none",
        "normalization": "RGB/255 then ImageNet mean/std",
        "positive_raw_mask_value": 0,
        "bn_calibrated": True, "save_every_epoch": SAVE_EVERY_EPOCH,
        "bn_calibration_data": "Train", "bn_calibration_batch_size": BATCH_SIZE,
        "bn_calibration_shuffle": False,
        "model_info": MODEL_INFO, "versions": VERSIONS,
        "test_checkpoint_selection": "fixed_final_epoch", "test_epoch": EPOCHS,
    }
    report_path = OUTPUT_DIR / "training_report.json"
    if FINAL_PATH.exists() and BEST_PATH.exists() and HISTORY_PATH.exists():
        old_config = json.loads((OUTPUT_DIR / "config.json").read_text(encoding="utf-8"))
        if any(old_config.get(key) != value for key, value in config.items()):
            raise RuntimeError("The saved run has different settings. Select another OUTPUT_DIR.")
        old_history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
        final = torch.load(FINAL_PATH, map_location="cpu", weights_only=True)
        complete = len(old_history) == EPOCHS and final["epoch"] == EPOCHS
        if SAVE_EVERY_EPOCH:
            complete = complete and all(
                (EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth").is_file()
                for epoch in range(1, EPOCHS + 1)
            )
        if (complete and final.get("bn_calibrated") is True
                and final.get("architecture") == MODEL_NAME and report_path.is_file()):
            best_record = max(old_history, key=lambda row: row["validation"]["iou"])
            print(f"All {EPOCHS} epochs are complete; training will not be repeated.")
            print("Best epoch:", best_record["epoch"])
            return old_history
    if WEIGHTS_DIR.exists() and any(WEIGHTS_DIR.rglob("*.pth")):
        raise RuntimeError("An incomplete or different run exists. Select another OUTPUT_DIR; existing weights will not be overwritten.")
    for path in (WEIGHTS_DIR, EPOCH_WEIGHTS_DIR, PREDICTION_DIR, MASK_DIR, CONFUSION_DIR):
        path.mkdir(parents=True, exist_ok=True)
    if SAVE_EVERY_EPOCH and shutil.disk_usage(OUTPUT_DIR).free < 6 * 1024**3:
        raise RuntimeError("At least 6 GiB of free disk space is required for checkpoints and visualizations.")
    save_json(OUTPUT_DIR / "config.json", config)
    set_seed(TRAINING_SEED)
    loaders = make_loaders(DATASETS[EXPERIMENT_NAME])
    calibration_loader = DataLoader(
        DATASETS[EXPERIMENT_NAME]["Train"], batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=False,
    )
    model = build_model(pretrained=True).to(DEVICE)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE)
    best_iou, history = -1.0, []
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
    started = time.perf_counter()
    try:
        for epoch in range(1, EPOCHS + 1):
            epoch_started = time.perf_counter()
            train_stats = run_epoch(model, loaders["Train"], optimizer)
            normal_val = run_epoch(model, loaders["Validation"])

            rng_devices = [DEVICE] if DEVICE.type == "cuda" else []
            with torch.random.fork_rng(devices=rng_devices):
                calibrated_model = recalibrate_bn_copy(model, calibration_loader)
                val_stats = run_epoch(calibrated_model, loaders["Validation"])
            history.append({
                "epoch": epoch, "train": train_stats, "normal_validation": normal_val, "validation": val_stats,
                "epoch_seconds": time.perf_counter() - epoch_started,
            })
            if val_stats["iou"] > best_iou:
                best_iou = val_stats["iou"]
                save_model_checkpoint(BEST_PATH, calibrated_model, epoch, val_stats, best_iou)
            if SAVE_EVERY_EPOCH:
                save_model_checkpoint(
                    EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth", calibrated_model, epoch, val_stats, best_iou
                )
            if epoch == EPOCHS:
                save_model_checkpoint(FINAL_PATH, calibrated_model, epoch, val_stats, best_iou)
            write_training_history(history)
            print(f"Epoch {epoch:02d}/{EPOCHS} | Train loss={train_stats['loss']:.4f}, "
                  f"IoU={train_stats['iou']:.4f} | "
                  f"Validation loss={val_stats['loss']:.4f}, "
                  f"IoU={val_stats['iou']:.4f}, F1={val_stats['f1']:.4f}")
            del calibrated_model
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        elapsed = time.perf_counter() - started
        best_record = max(history, key=lambda row: row["validation"]["iou"])
        report = {
            "architecture": MODEL_NAME, "experiment": EXPERIMENT_NAME,
            "status": "completed", "completed_epochs": len(history),
            "selected_epoch": EPOCHS, "best_validation_epoch": best_record["epoch"],
            "test_checkpoint_selection": "fixed_final_epoch",
            "test_epoch": EPOCHS, "test_checkpoint": str(FINAL_PATH),
            "best_validation": best_record["validation"], "bn_calibrated": True,
            "normal_validation_at_selected_epoch": best_record["normal_validation"],
            "parameters": MODEL_INFO["parameters"],
            "trainable_parameters": MODEL_INFO["trainable_parameters"],
            "training_seconds": elapsed, "training_minutes": elapsed / 60,
            "time_includes": "train + normal validation + BN calibration + calibrated validation + checkpoint/CSV writes; excludes test/plots/download",
            "batch_size": BATCH_SIZE, "evaluation_batch_size": EVAL_BATCH_SIZE,
            "versions": VERSIONS,
            "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU",
            "peak_allocated_gpu_gib": torch.cuda.max_memory_allocated(DEVICE) / 1024**3 if DEVICE.type == "cuda" else None,
            "training_exception": None,
            "validation_iou_below_0_5_epochs": [r["epoch"] for r in history if r["validation"]["iou"] < 0.5],
            "normal_validation_iou_below_0_5_epochs": [r["epoch"] for r in history if r["normal_validation"]["iou"] < 0.5],
        }
        save_json(report_path, report)
    except (Exception, KeyboardInterrupt) as error:
        save_json(OUTPUT_DIR / "training_problem.json", {
            "architecture": MODEL_NAME, "completed_epochs": len(history),
            "exception_type": type(error).__name__, "message": str(error),
            "elapsed_seconds": time.perf_counter() - started,
        })
        raise
    finally:
        if "calibrated_model" in locals():
            del calibrated_model
        del model, optimizer
        torch.cuda.empty_cache()
    print("Best checkpoint:", BEST_PATH)
    print("Final checkpoint:", FINAL_PATH)
    print("Train logs:", TRAIN_CSV_PATH)
    print("Training report:", report_path)
    print(f"Approximate training time: {elapsed / 60:.2f} minutes")
    return history


In [ ]:
# Train for 50 epochs

training_history = train_calibrated_experiment()


In [ ]:
# Plot training and validation metrics

with TRAIN_CSV_PATH.open(newline="", encoding="utf-8") as file:
    graph_rows = list(csv.DictReader(file))
graph_epochs = [int(row["epoch"]) for row in graph_rows]
panels = [
    ("loss", "Loss"), ("iou_score", "IoU"), ("fscore", "F1-score"),
    ("accuracy", "Accuracy"), ("precision", "Precision"), ("recall", "Recall"),
]


def draw_metric_panels(items, filename, columns, show=False):
    rows = (len(items) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(5.3 * columns, 4 * rows), squeeze=False)
    for ax, (key, title) in zip(axes.flat, items):
        ax.plot(graph_epochs, [float(row[key]) for row in graph_rows], label="Train")
        ax.plot(graph_epochs, [float(row["val_" + key]) for row in graph_rows], label="Validation")
        ax.set(title=title, xlabel="Epoch", ylabel=title)
        ax.grid(True, alpha=0.3)
        ax.legend()
    for ax in list(axes.flat)[len(items):]:
        ax.set_visible(False)
    fig.suptitle("V2 - U-Net (MiT-B2)")
    fig.tight_layout()
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print("Saved:", path)


draw_metric_panels(panels, "metric_graph.png", 3, show=True)
draw_metric_panels(panels[:2], "metric_graph_loss_iou.png", 2)
draw_metric_panels(panels[2:4], "metric_graph_f1_acc.png", 2)
draw_metric_panels(panels[4:], "metric_graph_pre_recall.png", 2)


In [ ]:
# Define test and prediction export

def cnn_confusion_image(target, prediction):
    # Preserve the reference CNN confusion-map colors.
    output = np.zeros((*target.shape, 3), dtype=np.uint8)
    output[(target == 0) & (prediction == 0)] = (0, 255, 0)
    output[(target == 0) & (prediction == 1)] = (0, 0, 255)
    output[(target == 1) & (prediction == 0)] = (255, 0, 0)
    return output


def save_prediction_visual(name, image_rgb, target, prediction, confusion, metrics):
    path = PREDICTION_DIR / (Path(name).stem + ".png")
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    axes[0].imshow(image_rgb)
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[3].imshow(confusion)
    for ax, title in zip(axes, ("Image", "Ground truth", "Prediction", "Confusion (CNN palette)")):
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{name} | IoU={metrics['iou']:.4f} | F1={metrics['f1']:.4f}")
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def evaluate_and_export_test():
    if not FINAL_PATH.is_file() or not BEST_PATH.is_file():
        raise RuntimeError("Complete the 50-epoch training run first.")
    saved = torch.load(FINAL_PATH, map_location="cpu", weights_only=True)
    if (saved.get("bn_calibrated") is not True or saved.get("experiment") != EXPERIMENT_NAME
            or saved.get("architecture") != MODEL_NAME):
        raise RuntimeError("A calibrated checkpoint from the selected experiment is required.")
    if saved.get("epoch") != EPOCHS:
        raise RuntimeError("Test requires the calibrated final checkpoint from epoch 50.")
    dataset = DATASETS[EXPERIMENT_NAME]["Test"]
    stems = [Path(name).stem for name in dataset.names]
    if len(stems) != len(set(stems)):
        raise RuntimeError("Test filenames contain duplicate stems.")
    summary_path = OUTPUT_DIR / "test_summary.json"
    fingerprint = checkpoint_hash(FINAL_PATH)
    if summary_path.exists():
        previous = json.loads(summary_path.read_text(encoding="utf-8"))
        if (previous.get("checkpoint_sha256") != fingerprint
                or previous.get("selected_epoch") != EPOCHS
                or previous.get("bn_calibrated") is not True
                or previous.get("checkpoint_selection") != "fixed_final_epoch"):
            raise RuntimeError("The saved test results belong to another checkpoint and will not be overwritten.")
        outputs_complete = all(
            (OUTPUT_DIR / filename).is_file()
            for filename in ("test_summary.csv", "test_per_patch.csv")
        ) and all(
            (PREDICTION_DIR / f"{stem}.png").is_file()
            and (MASK_DIR / f"{stem}.tif").is_file()
            and (CONFUSION_DIR / f"{stem}.png").is_file()
            for stem in stems
        )
        if not outputs_complete:
            raise RuntimeError("Test results exist, but some prediction files are missing.")
        print("Test results and predictions already exist; inference will not be repeated.")
        return previous
    for path in (PREDICTION_DIR, MASK_DIR, CONFUSION_DIR):
        path.mkdir(parents=True, exist_ok=True)
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_model = build_model(pretrained=False).to(DEVICE)
    test_model.load_state_dict(saved["model_state_dict"], strict=True)
    test_model.eval()
    rows = []
    per_patch_path = OUTPUT_DIR / "test_per_patch.csv"
    try:
        with per_patch_path.open("w", newline="", encoding="utf-8") as file:
            writer = None
            with torch.no_grad():
                for index, (images, masks) in enumerate(loader):
                    images = images.to(DEVICE, non_blocking=True)
                    masks = masks.to(DEVICE, non_blocking=True)
                    logits = test_model(images)
                    values = {name: value[0].item() for name, value in patch_metrics(logits, masks).items()}
                    row = {"filename": dataset.names[index],
                           "loss": dice_loss_with_logits(logits, masks).item(), **values}
                    target = (masks[0, 0].cpu().numpy() >= 0.5).astype(np.uint8)
                    prediction = (torch.sigmoid(logits)[0, 0].cpu().numpy() >= THRESHOLD).astype(np.uint8)
                    row.update(
                        tp=int(np.sum((target == 1) & (prediction == 1))),
                        fp=int(np.sum((target == 0) & (prediction == 1))),
                        fn=int(np.sum((target == 1) & (prediction == 0))),
                        tn=int(np.sum((target == 0) & (prediction == 0))),
                    )
                    name = dataset.names[index]
                    stem = Path(name).stem
                    image_bgr = cv2.imread(str(dataset.images[name]), cv2.IMREAD_COLOR)
                    if image_bgr is None:
                        raise RuntimeError(f"Cannot read image: {dataset.images[name]}")
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    confusion = cnn_confusion_image(target, prediction)
                    if not cv2.imwrite(str(MASK_DIR / f"{stem}.tif"), prediction):
                        raise RuntimeError(f"Cannot save prediction mask: {name}")
                    if not cv2.imwrite(str(CONFUSION_DIR / f"{stem}.png"),
                                       cv2.cvtColor(confusion, cv2.COLOR_RGB2BGR)):
                        raise RuntimeError(f"Cannot save confusion map: {name}")
                    save_prediction_visual(name, image_rgb, target, prediction, confusion, values)
                    if writer is None:
                        writer = csv.DictWriter(file, fieldnames=list(row))
                        writer.writeheader()
                    writer.writerow(row)
                    file.flush()
                    rows.append(row)
                    if index == 0 or (index + 1) % 25 == 0 or index + 1 == len(dataset):
                        print(f"Test/prediction: {index + 1}/{len(dataset)}")
    finally:
        del test_model
        torch.cuda.empty_cache()
    if len(rows) != len(dataset):
        raise RuntimeError("Not all test patches were processed.")
    summary = {
        "experiment": EXPERIMENT_NAME, "architecture": MODEL_NAME, "selected_epoch": saved["epoch"],
        "checkpoint": str(FINAL_PATH), "checkpoint_sha256": fingerprint,
        "bn_calibrated": True, "test_patches": len(rows),
        "checkpoint_selection": "fixed_final_epoch",
        "bn_calibration_data": "Train",
        "test": {key: float(np.mean([row[key] for row in rows]))
                 for key in ("loss", *METRIC_NAMES)},
    }
    save_json(summary_path, summary)
    write_rows(OUTPUT_DIR / "test_summary.csv", [{
        "experiment": EXPERIMENT_NAME, "architecture": MODEL_NAME, "selected_epoch": saved["epoch"],
        "test_patches": len(rows), **summary["test"],
    }])
    return summary


In [ ]:
# Test the final checkpoint

test_summary = evaluate_and_export_test()
print("Test checkpoint:", test_summary["checkpoint"])
print("Test epoch:", test_summary["selected_epoch"])
print("Test patches:", test_summary["test_patches"])
for name, value in test_summary["test"].items():
    print(f"{name}: {value:.4f}")
print("Test summary:", OUTPUT_DIR / "test_summary.json")
print("Prediction:", PREDICTION_DIR)
print("Masks:", MASK_DIR)
print("Confusion:", CONFUSION_DIR)


In [ ]:
# Verify saved outputs

required_files = [BEST_PATH, FINAL_PATH, HISTORY_PATH, TRAIN_CSV_PATH,
                  OUTPUT_DIR / "config.json", OUTPUT_DIR / "training_report.json",
                  OUTPUT_DIR / "metric_graph.png", OUTPUT_DIR / "metric_graph_loss_iou.png",
                  OUTPUT_DIR / "metric_graph_f1_acc.png", OUTPUT_DIR / "metric_graph_pre_recall.png",
                  OUTPUT_DIR / "test_per_patch.csv", OUTPUT_DIR / "test_summary.json",
                  OUTPUT_DIR / "test_summary.csv"]
required_files.extend(EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth"
                      for epoch in range(1, EPOCHS + 1))
test_names = DATASETS[EXPERIMENT_NAME]["Test"].names
for name in test_names:
    stem = Path(name).stem
    required_files.extend((PREDICTION_DIR / f"{stem}.png", MASK_DIR / f"{stem}.tif",
                           CONFUSION_DIR / f"{stem}.png"))
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError("Missing files:\n" + "\n".join(missing[:15]))

history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
report = json.loads((OUTPUT_DIR / "training_report.json").read_text(encoding="utf-8"))
summary = json.loads((OUTPUT_DIR / "test_summary.json").read_text(encoding="utf-8"))
if ([row["epoch"] for row in history] != list(range(1, EPOCHS + 1))
        or report.get("status") != "completed" or report.get("completed_epochs") != EPOCHS):
    raise RuntimeError("The saved training history or report is incomplete.")
with TRAIN_CSV_PATH.open(newline="", encoding="utf-8") as file:
    training_rows = list(csv.DictReader(file))
with (OUTPUT_DIR / "test_per_patch.csv").open(newline="", encoding="utf-8") as file:
    test_rows = list(csv.DictReader(file))
if ([int(row["epoch"]) for row in training_rows] != list(range(1, EPOCHS + 1))
        or [row["filename"] for row in test_rows] != test_names):
    raise RuntimeError("The training or per-patch CSV does not contain all expected rows.")
if (summary.get("selected_epoch") != EPOCHS or summary.get("bn_calibrated") is not True
        or summary.get("checkpoint_selection") != "fixed_final_epoch"
        or summary.get("checkpoint") != str(FINAL_PATH)
        or summary.get("checkpoint_sha256") != checkpoint_hash(FINAL_PATH)
        or summary.get("test_patches") != len(test_names)):
    raise RuntimeError("The test summary does not match the calibrated final checkpoint.")

for epoch in range(1, EPOCHS + 1):
    path = EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth"
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if (saved.get("epoch") != epoch or saved.get("bn_calibrated") is not True
            or saved.get("experiment") != EXPERIMENT_NAME):
        raise RuntimeError(f"Invalid calibrated epoch checkpoint: {path}")
    del saved
best = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
best_record = max(history, key=lambda row: row["validation"]["iou"])
if best.get("epoch") != best_record["epoch"] or best.get("bn_calibrated") is not True:
    raise RuntimeError("The best checkpoint does not match calibrated validation selection.")
del best
final = torch.load(FINAL_PATH, map_location="cpu", weights_only=True)
epoch_50 = torch.load(EPOCH_WEIGHTS_DIR / f"epoch_{EPOCHS:03d}.pth",
                      map_location="cpu", weights_only=True)
if (final.get("epoch") != EPOCHS or final.get("bn_calibrated") is not True
        or final.get("experiment") != EXPERIMENT_NAME
        or final["model_state_dict"].keys() != epoch_50["model_state_dict"].keys()
        or any(not torch.equal(tensor, epoch_50["model_state_dict"][name])
               for name, tensor in final["model_state_dict"].items())):
    raise RuntimeError("Final weights and the calibrated epoch-50 weights do not match.")
del final, epoch_50
print("Verified: best + final + all 50 epoch checkpoints.")
print("Verified: all graphs, CSV/JSON reports, predictions, masks, and confusion maps.")
print("Test epoch:", summary["selected_epoch"])
print("Files per test visualization type:", len(test_names))
print("Output:", OUTPUT_DIR)
